In [ ]:
import json, re
from pathlib import Path
from tqdm import tqdm
import fitz  # PyMuPDF
import pandas as pd
from openai import OpenAI

In [ ]:
#! pip install tools fitz openai
#!pip install --upgrade --force-reinstall pymupdf

In [ ]:
PDF_PATH = "/content/chung2020chamorro.pdf"
MODEL = "gpt-5.2"
MAX_CHARS = 36000
OVERLAP_PAGES = 1

In [ ]:
OUT_JSONL = "/content/chung2020_parallel_extracted.jsonl"
OUT_CSV   = "/content/chung2020_parallel_extracted.csv"

OPENAI_API_KEY=YOUR_API_HERE


In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)
from tqdm.auto import tqdm

In [ ]:

SYSTEM_PROMPT = r"""
You are extracting parallel data from Sandra Chung (2020) "Chamorro Grammar".
Your job is to identify and reconstruct three kinds of parallel examples:

(1) IGT BLOCKS (glossed examples):
Usually formatted as:
(14) a. <Chamorro line(s)>
     <gloss line(s) with Leipzig tags like AGR, PROG, COMP, UNM, etc.>
     ‘<English free translation>’
     (<source like Alamagan 16>)
Subexamples a/b/c are common. The Chamorro line may wrap across lines.
Gloss may wrap across multiple lines.
The translation may be on a separate line and may be followed by a source line in parentheses.

(2) INLINE EXAMPLES:
Object-language phrase/sentence embedded in running prose, with an English translation nearby
(often in quotes). Extract only when there is an unambiguous Chamorro ↔ English pairing.

(3) TRANSLATED TEXTS:
Running Chamorro text with an English translation section. Align sentence-by-sentence if possible,
else align paragraph-by-paragraph.

OUTPUT
Return STRICT JSON ONLY (no markdown), as a list of objects.
Each object must contain:
- "type": "igt" | "inline" | "text"
- "example_id": string like "14a", "14b", "14c" if available; else null
- "object_language": Chamorro text ONLY (no gloss tags, no English, no Leipzig abbreviations)
- "gloss": gloss line(s) ONLY if type="igt", else null
- "translation": English free translation ONLY
- "source_ref": source/citation in parentheses like "(Alamagan 16)" if present, else null
- "page_hint": page label if present in the excerpt (e.g., "=== PAGE 39 ==="), else null
- "context_before": up to 1 sentence of surrounding explanatory prose immediately before the example (exclude other examples)
- "context_after": up to 1 sentence of surrounding explanatory prose immediately after the example (exclude other examples)

CRITICAL RULES (NO EXCEPTIONS)
- Do not include gloss leakage in "object_language". Remove tokens such as AGR, PROG, COMP, UNM, 1/2/3, SG/PL, etc.
- Do not include example labels "(14)", "a.", etc. inside object_language/gloss/translation. Put them in example_id only.
- If the Chamorro line is split across multiple lines, join them into one clean sentence.
- For IGT, ensure gloss is captured (non-empty). If gloss is missing, do NOT output as type="igt".
- Ensure every object has non-empty object_language and translation.
- Deduplicate: do not output two objects with identical (object_language, translation).

Now extract examples from the excerpt below.
""".strip()

def chunk_pages(doc, max_chars=MAX_CHARS, overlap_pages=OVERLAP_PAGES):
    pages_text = [doc.load_page(i).get_text("text") or "" for i in range(doc.page_count)]
    i = 0
    while i < doc.page_count:
        start = i
        end = i
        buff = ""
        while end < doc.page_count and len(buff) + len(pages_text[end]) < max_chars:
            buff += f"\n\n=== PAGE {end+1} ===\n" + pages_text[end]
            end += 1

        ov_start = max(0, start - overlap_pages)
        ov_buff = ""
        for p in range(ov_start, start):
            ov_buff += f"\n\n=== PAGE {p+1} ===\n" + pages_text[p]

        yield (ov_start + 1, end, ov_buff + buff)  # 1-indexed
        i = end

def parse_json_list(text: str):
    text = text.strip()
    if text.startswith("["):
        return json.loads(text)
    a = text.find("[")
    b = text.rfind("]")
    if a != -1 and b != -1 and b > a:
        return json.loads(text[a:b+1])
    raise ValueError("Model did not return a JSON list.")

# minimal “leakage guard” (NOT a parser): just rejects obvious bad outputs
LEAK = re.compile(r"\b(AGR|PROG|COMP|UNM|SG|PL|PST|PRF|ERG|ABS|DAT|GEN|LOC|INSTR|NMLZ)\b")


In [ ]:

doc = fitz.open(PDF_PATH)

rows = []
seen = set()

chunks = list(chunk_pages(doc))

for page_start, page_end, chunk_text in tqdm(
        chunks,
        desc="Extracting Chung2020 examples",
        unit="chunk"
    ):


    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": chunk_text},
        ],
        # temperature=0,
    )
    items = parse_json_list(resp.output_text)

    for it in items:
        if not isinstance(it, dict):
            continue
        obj = (it.get("object_language") or "").strip()
        trn = (it.get("translation") or "").strip()

        if not obj or not trn:
            continue
        if LEAK.search(obj):
            continue  # reject leaked-gloss object-language

        key = (it.get("type"), obj, trn)
        if key in seen:
            continue
        seen.add(key)

        if not it.get("page_hint"):
            it["page_hint"] = f"{page_start}-{page_end}"

        rows.append(it)

# Save
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print("Extracted:", len(df))
display(df.head(30))
print("Saved:", OUT_JSONL)
print("Saved:", OUT_CSV)


Extracting Chung2020 examples:   0%|          | 0/44 [00:00<?, ?chunk/s]

Extracted: 2414


,type,example_id,object_language,gloss,translation,source_ref,page_hint,context_before,context_after
0,igt,1a,Kumåkati.,AGR.cry.PROG,S/he is crying.,None,=== PAGE 34 ===,The predicate can be from any of the major par...,None
1,igt,1b,Hu po’lu.,AGR put,I put it (there).,None,=== PAGE 34 ===,The predicate can be from any of the major par...,None
2,igt,2a,Manlokka’.,AGR.tall,They are tall.,None,=== PAGE 34 ===,The predicate can be from any of the major par...,None
3,igt,2b,Agaga’.,AGR.red,It is red.,None,=== PAGE 34 ===,The predicate can be from any of the major par...,None
4,igt,3a,Ma’estru.,teacher,He is a teacher.,None,=== PAGE 34 ===,The predicate can be from any of the major par...,None
5,igt,3b,Ga’lågu.,dog,It’s a dog.,None,=== PAGE 34 ===,The predicate can be from any of the major par...,The predicate can also be a preposition (but s...
6,igt,4,Para månu?,to where?,Where are they heading?,None,=== PAGE 35 ===,The predicate can also be a preposition (but s...,Predicates that are verbs or adjectives agree ...
7,igt,5a,Kumåkati i neni.,AGR.cry.PROG the baby,The baby is crying.,None,=== PAGE 35 ===,Chamorro is a predicate-first language.,None
8,igt,5b,Hu po’lu i nengkanu’ gi hilu’ lamasa.,AGR put the food LCL top.L table,I put the food on top of the table.,None,=== PAGE 35 ===,Chamorro is a predicate-first language.,None
9,igt,5c,Manlokka’ siha.,AGR.tall they,They are tall.,None,=== PAGE 35 ===,Chamorro is a predicate-first language.,None


Saved: /content/chung2020_parallel_extracted.jsonl
Saved: /content/chung2020_parallel_extracted.csv


In [ ]:
df_small = df[["type","example_id","object_language","translation","source_ref","page_hint"]]
display(df_small.head(50))


,type,example_id,object_language,translation,source_ref,page_hint
0,igt,1a,Kumåkati.,S/he is crying.,None,=== PAGE 34 ===
1,igt,1b,Hu po’lu.,I put it (there).,None,=== PAGE 34 ===
2,igt,2a,Manlokka’.,They are tall.,None,=== PAGE 34 ===
3,igt,2b,Agaga’.,It is red.,None,=== PAGE 34 ===
4,igt,3a,Ma’estru.,He is a teacher.,None,=== PAGE 34 ===
5,igt,3b,Ga’lågu.,It’s a dog.,None,=== PAGE 34 ===
6,igt,4,Para månu?,Where are they heading?,None,=== PAGE 35 ===
7,igt,5a,Kumåkati i neni.,The baby is crying.,None,=== PAGE 35 ===
8,igt,5b,Hu po’lu i nengkanu’ gi hilu’ lamasa.,I put the food on top of the table.,None,=== PAGE 35 ===
9,igt,5c,Manlokka’ siha.,They are tall.,None,=== PAGE 35 ===


In [ ]:
PDF_PATH = "/content/emenanjo2015grammar.pdf"
OUT_JSONL = "/content/emenanjo2015_parallel_extracted.jsonl"
OUT_CSV   = "/content/emenanjo2015_parallel_extracted.csv"

In [ ]:
doc = fitz.open(PDF_PATH)

rows = []
seen = set()

chunks = list(chunk_pages(doc))

for page_start, page_end, chunk_text in tqdm(
        chunks,
        desc="Extracting emenanjo2015 examples",
        unit="chunk"
    ):

    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": chunk_text},
        ],
        # temperature=0,
    )
    try:
      items = parse_json_list(resp.output_text)

      for it in items:
          if not isinstance(it, dict):
              continue

          obj = (it.get("object_language") or "").strip()
          trn = (it.get("translation") or "").strip()

          if not obj or not trn:
              continue
          if LEAK.search(obj):
              continue

          key = (it.get("type"), obj, trn)
          if key in seen:
              continue
          seen.add(key)

          if not it.get("page_hint"):
              it["page_hint"] = f"{page_start}-{page_end}"

          rows.append(it)
    except:
      pass



# Save
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print("Extracted:", len(df))
display(df.head(30))
print("Saved:", OUT_JSONL)
print("Saved:", OUT_CSV)

Extracting emenanjo2015 examples:   0%|          | 0/38 [00:00<?, ?chunk/s]

Extracted: 1094


,type,example_id,object_language,gloss,translation,source_ref,page_hint,context_before,context_after
0,inline,None,oke,None,male,None,=== PAGE 59 ===,In noun class languages as those found in Afri...,"The use of oke ‘male’, nwunyè ‘female’ to mark..."
1,inline,None,nwunyè,None,female,None,=== PAGE 59 ===,In noun class languages as those found in Afri...,"The use of oke ‘male’, nwunyè ‘female’ to mark..."
2,inline,None,uri/akwukwo mmuo,None,different indigenous writing systems,None,=== PAGE 69 ===,Different indigenous writing systems called ur...,But the orthography first used in a pan-Igbo w...
3,inline,None,Òtu Sùbakwanù Ìgbò,None,Society for Promoting Igbo Language and Cultur...,None,=== PAGE 72 ===,the emergence of learned societies like the Ig...,All of the above are designed to complement th...
4,inline,None,bùre-è-kì,None,break,None,=== PAGE 85 ===,In the unvarnished speech of the Type A rustic...,In view of the restrictions observed above abo...
5,inline,None,bì-re-è-kì,None,break,None,=== PAGE 85 ===,In the unvarnished speech of the Type A rustic...,In view of the restrictions observed above abo...
6,inline,None,sùkulù,None,school,None,=== PAGE 85 ===,In the unvarnished speech of the Type A rustic...,In view of the restrictions observed above abo...
7,inline,None,sùkuùlù,None,school,None,=== PAGE 85 ===,In the unvarnished speech of the Type A rustic...,In view of the restrictions observed above abo...
8,inline,None,bo͠tù͠lù͠,None,bottle,None,=== PAGE 85 ===,In the unvarnished speech of the Type A rustic...,In view of the restrictions observed above abo...
9,inline,None,ka-pi͠-ti͠-nʌ,None,captain,None,=== PAGE 85 ===,In the unvarnished speech of the Type A rustic...,In view of the restrictions observed above abo...


Saved: /content/emenanjo2015_parallel_extracted.jsonl
Saved: /content/emenanjo2015_parallel_extracted.csv


In [ ]:
PDF_PATH = "/content/hewitt1995georgian.pdf"
OUT_JSONL = "/content/hewitt1995georgian_parallel_extracted.jsonl"
OUT_CSV   = "/content/hewitt1995georgian_parallel_extracted.csv"

doc = fitz.open(PDF_PATH)

rows = []
seen = set()

chunks = list(chunk_pages(doc))

for page_start, page_end, chunk_text in tqdm(
        chunks,
        desc="Extracting emenanjo2015 examples",
        unit="chunk"
    ):

    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": chunk_text},
        ],
        # temperature=0,
    )
    try:

      items = parse_json_list(resp.output_text)

      for it in items:
          if not isinstance(it, dict):
              continue

          obj = (it.get("object_language") or "").strip()
          trn = (it.get("translation") or "").strip()

          if not obj or not trn:
              continue
          if LEAK.search(obj):
              continue

          key = (it.get("type"), obj, trn)
          if key in seen:
              continue
          seen.add(key)

          if not it.get("page_hint"):
              it["page_hint"] = f"{page_start}-{page_end}"

          rows.append(it)
    except:
      pass



# Save
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print("Extracted:", len(df))
display(df.head(30))
print("Saved:", OUT_JSONL)
print("Saved:", OUT_CSV)

Extracting emenanjo2015 examples:   0%|          | 0/42 [00:00<?, ?chunk/s]

Extracted: 1105


,type,example_id,object_language,gloss,translation,source_ref,page_hint,context_before,context_after
0,inline,None,kart-v-el-i,None,Georgian,None,=== PAGE 29 ===,The origin of this confused thinking is perhap...,A Georgian calls himself kart-v-el-i (cf. sa-k...
1,inline,None,sa-kart-v-el-o,None,Georgia,None,=== PAGE 29 ===,The origin of this confused thinking is perhap...,A Georgian calls himself kart-v-el-i (cf. sa-k...
2,inline,None,kart-ul-i,None,Georgian (non-human),None,=== PAGE 29 ===,The origin of this confused thinking is perhap...,The Georgian expression for 'Kartvelian langua...
3,inline,None,kart-v-el-ur-i en-eb-i,None,Kartvelian languages,None,=== PAGE 29 ===,A Georgian calls himself kart-v-el-i (cf. sa-k...,"Regrettably, the language lacks the equivalent..."
4,inline,None,šušanik'is c'ameba,None,Martyrdom of St. Shushanik,None,=== PAGE 27 ===,The first native work of Georgian literature (...,The oldest dated manuscript thus far discovere...
5,inline,None,vepxist'q'aosani,None,The Man in the Panther's Skin,None,=== PAGE 28 ===,The most famous work of the mediaeval period i...,"It consists of 1598 quatrains, end-rhyming for..."
6,inline,None,kartlis cxovreba,None,History of Georgia,None,=== PAGE 28 ===,Mention should also be made of the chronicles ...,The three most important literary figures of t...
7,igt,None,ga+mo+m+cxv+ar-i,baked-AGR,baked,None,=== PAGE 34 ===,"For example, we could gloss each component of ...",To illustrate the second case let us take the ...
8,inline,None,sxiv-osan-i,None,spreading rays,None,=== PAGE 44 ===,That we are here dealing with morpho-phonemic ...,and sxiv-ur-i energia 'ray-energy/electro-magn...
9,inline,None,sxiv-ur-i energia,None,ray-energy/electro-magnetic energy,None,=== PAGE 44 ===,That we are here dealing with morpho-phonemic ...,"Indeed, the epenthetic rules outlined above ar..."


Saved: /content/hewitt1995georgian_parallel_extracted.jsonl
Saved: /content/hewitt1995georgian_parallel_extracted.csv


In [ ]:
PDF_PATH = "/content/hualde2003grammar.pdf"
OUT_JSONL = "/content/hualde2003_parallel_extracted.jsonl"
OUT_CSV   = "/content/hualde2003_parallel_extracted.csv"

doc = fitz.open(PDF_PATH)

rows = []
seen = set()

chunks = list(chunk_pages(doc))

for page_start, page_end, chunk_text in tqdm(
        chunks,
        desc="Extracting emenanjo2015 examples",
        unit="chunk"
    ):

    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": chunk_text},
        ],
        # temperature=0,
    )
    try:
      items = parse_json_list(resp.output_text)

      for it in items:
          if not isinstance(it, dict):
              continue

          obj = (it.get("object_language") or "").strip()
          trn = (it.get("translation") or "").strip()

          if not obj or not trn:
              continue
          if LEAK.search(obj):
              continue

          key = (it.get("type"), obj, trn)
          if key in seen:
              continue
          seen.add(key)

          if not it.get("page_hint"):
              it["page_hint"] = f"{page_start}-{page_end}"

          rows.append(it)
    except:
      pass


# Save
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print("Extracted:", len(df))
display(df.head(30))
print("Saved:", OUT_JSONL)
print("Saved:", OUT_CSV)

Extracting emenanjo2015 examples:   0%|          | 0/50 [00:00<?, ?chunk/s]

Extracted: 3077


,type,example_id,object_language,gloss,translation,source_ref,page_hint,context_before,context_after
0,inline,None,euskara,None,Basque (the Basques’ name for their language).,None,=== PAGE 31 ===,"In post-Renaissance Spain, the name vizcaino, ...",Euskara is opposed to erdara (or erdera) 'fore...
1,inline,None,erdara,None,foreign language.,None,=== PAGE 31 ===,The Basques call their language euskara (and d...,Both words have an ending which very likely de...
2,inline,None,ibilera,None,way of walking.,None,=== PAGE 31 ===,Both words have an ending which very likely de...,In erdara the first element appears to be erdi...
3,inline,None,Euskal Herria,None,country of the Basque language.,None,=== PAGE 32 ===,Basque speakers refer to the land where their ...,"In the first decades of the 20th century, the ..."
4,inline,None,euskaldunak,None,those who have the Basque language.,None,=== PAGE 32 ===,Basque speakers refer to the land where their ...,"In the first decades of the 20th century, the ..."
5,inline,None,euzkera,None,a respelling of 'euskara' (used in the early 2...,None,=== PAGE 32 ===,"In the first decades of the 20th century, the ...",Related coinages of that vintage are the words...
6,inline,None,Euzkadi,None,Basque Country.,None,=== PAGE 32 ===,"In the first decades of the 20th century, the ...",The truly great and very influential Basque li...
7,inline,None,euzko,None,"Basque, in an ethnic or political sense.",None,=== PAGE 32 ===,Related coinages of that vintage are the words...,The truly great and very influential Basque li...
8,inline,None,euzkotar,None,Basque citizen; ethnic Basque.,None,=== PAGE 32 ===,Related coinages of that vintage are the words...,The truly great and very influential Basque li...
9,inline,None,Eusko jaurlaritza,None,Basque government.,None,=== PAGE 32 ===,His wish was to avoid the misuse of words like...,He succesfully campaigned for terms like Eusko...


Saved: /content/hualde2003_parallel_extracted.jsonl
Saved: /content/hualde2003_parallel_extracted.csv


In [ ]:
t

In [ ]:
PDF_PATH = "/content/martin1992reference.pdf"
OUT_JSONL = "/content/martin1992_parallel_extracted.jsonl"
OUT_CSV   = "/content/martin1992_parallel_extracted.csv"

doc = fitz.open(PDF_PATH)

rows = []
seen = set()

chunks = list(chunk_pages(doc))

for page_start, page_end, chunk_text in tqdm(
        chunks,
        desc="Extracting emenanjo2015 examples",
        unit="chunk"
    ):

    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": chunk_text},
        ],
        # temperature=0,
    )
    try:
      items = parse_json_list(resp.output_text)

      for it in items:
          if not isinstance(it, dict):
              continue

          obj = (it.get("object_language") or "").strip()
          trn = (it.get("translation") or "").strip()

          if not obj or not trn:
              continue
          if LEAK.search(obj):
              continue

          key = (it.get("type"), obj, trn)
          if key in seen:
              continue
          seen.add(key)

          if not it.get("page_hint"):
              it["page_hint"] = f"{page_start}-{page_end}"

          rows.append(it)
    except:
      pass


# Save
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print("Extracted:", len(df))
display(df.head(30))
print("Saved:", OUT_JSONL)
print("Saved:", OUT_CSV)

Extracting emenanjo2015 examples:   0%|          | 0/112 [00:00<?, ?chunk/s]

Extracted: 513


,type,example_id,object_language,gloss,translation,source_ref,page_hint,context_before,context_after
0,inline,None,koki,None,meat,None,=== PAGE 16 ===,When a Korean citation is within an English se...,The macron is usctl to mlrk long vowels in nxr...
1,inline,None,rer 'time' - a'ni han te't ey,None,in a short while,None,=== PAGE 71 ===,A number of nouns suppress the high pitch on a...,For at least one word this holds for the genit...
2,inline,None,ti tpo'l ay,None,in this village,(1459 Wel 8:94a),=== PAGE 71 ===,Monosyllabic nouns which do not lose their acc...,'s/cwurn 'dream'.
3,inline,None,'mwo'h ay s,None,mountain,None,=== PAGE 71 ===,In the same environment (before the locative m...,"""swo,t 'deep inside' * 'swo.kay s (1481 Twusi ..."
4,inline,None,'nvol.h ay,None,to the beams,(1481 Twusi 7:5a),=== PAGE 71 ===,In the same environment (before the locative m...,Usually the double dot is retained: nwu n ey s...
5,inline,None,nwu n ey s 'tol,None,moonlight on the snow,(1482 Kum-sam 2:6lb),=== PAGE 71 ===,Usually the double dot is retained:,'mul s 'ko'z ay (1459 Wel 8:99a) 'at the water...
6,inline,None,'mul s 'ko'z ay,None,at the water's edge,(1459 Wel 8:99a),=== PAGE 71 ===,Usually the double dot is retained: nwu n ey s...,"f- s) ""i'I ey (1475 Nay 2:2:47b) 'in the event..."
7,inline,None,'wiy'h ey 'sye,None,in back,None,=== PAGE 71 ===,Usually the double dot is retained: ... 'wiy'h...,The modern dialects of Hamkyeng and Kyengsang ...
8,inline,None,'srvoy ku'lus'tiki 'yey s'swo.h i'lo,None,it is a mold for making metal vessels,(1465 Wen l:l:2:l8la),=== PAGE 72 ===,"The summative -'li is nonleniting, and that ac...","Interestingly, when the summative -'li started..."
9,inline,None,Salkem sallye cwii,None,Save me!,None,=== PAGE 89 ===,The last word of Salkem sallye cwii 'Save me!'...,Similar are I ke(s) pr-l 'Look at this!'and Ka...


Saved: /content/martin1992_parallel_extracted.jsonl
Saved: /content/martin1992_parallel_extracted.csv
